In [ ]:
from pathlib import Path

import torch
from lightning.pytorch import Trainer, seed_everything
from torchvision.utils import make_grid

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
import pandas as pd
from IPython.display import display

from radio_diffusion.data.mojave import MojaveDataModule
from radio_diffusion.diffusion.ddpm import GaussianDiffusion
from radio_diffusion.models.unet import CompactUNet
from radio_diffusion.training.module import DDPMModule

In [ ]:
img_size=256
batch_size=32

server_data_dir = Path(f"/cephfs/users/aaki/data/Mojave_I_{img_size}")

if server_data_dir.exists():  # running on the server
    image_dir = server_data_dir
    remote_source = None
else:  # running locally
    image_dir = Path("../data") / f"Mojave_I_{img_size}"
    remote_source = f"nuradio:{server_data_dir}"

data = MojaveDataModule(
    data_dir=image_dir,
    batch_size=batch_size,
    num_workers=4,
    val_fraction=0.1,
    test_fraction=0.1,
    seed=42,
    remote_source=remote_source,
)

data.prepare_data()

## Sample Images to test Forward Diffusion

In [ ]:
data.setup("fit")
train_loader = data.train_dataloader()
images = next(iter(train_loader))

## Forward diffusion

The schedule assigns a noise variance $\beta_t$ to each step. The cumulative factor is $\bar{\alpha}_t = \prod_{s=0}^{t}(1 - \beta_s)$. A noisy image is computed directly from the original image:

$$x_t = \sqrt{\bar{\alpha}_t}\,x_{\mathrm{clean}} + \sqrt{1 - \bar{\alpha}_t}\,\epsilon, \qquad \epsilon \sim \mathcal{N}(0, I).$$

`GaussianDiffusion` stores the schedule and precomputed coefficients. Its `add_noise()` method returns both the noisy images and the sampled noise. The noise serves as the prediction target during training. This follows equation (4) of the [DDPM paper](https://arxiv.org/abs/2006.11239), with zero-based schedule indices: index 0 is already slightly noisy.

In [ ]:
diffusion = GaussianDiffusion(
    num_steps=1000,
    beta_start=1e-4,
    beta_end=0.02,
)

## Images at increasing noise levels

Each row shows the same digit, beginning with the original image. The same Gaussian noise is reused across the selected timesteps to make the change in noise strength visible. Each column is calculated directly from the clean images.

Noisy values can extend beyond `[-1, 1]`. Clipping is applied only to the display copy; the forward diffusion output retains its full range.

In [ ]:
timesteps = torch.linspace(
    0, diffusion.num_steps - 1, 5, dtype=torch.long
)
trajectory, timesteps = diffusion.forward_trajectory(images[:4], timesteps)

display_images = ((trajectory.flatten(0, 1) + 1) / 2).clamp(0, 1)
grid = make_grid(display_images, nrow=trajectory.shape[1])

In [ ]:
preview = grid[0].cpu().numpy()

# Power-law normalization
norm = PowerNorm(gamma=0.5, vmin=0.45, vmax=1)

fig, ax = plt.subplots(figsize=(14, 10))

im = ax.imshow(preview, cmap="inferno", norm=norm)
fig.colorbar(im, ax=ax, label="Normalized pixel intensity")

ax.axis("off")
None

## Model training

The U-Net is trained to recover the Gaussian noise used to construct each noisy image. `DDPMModule` samples a diffusion timestep for every image, applies forward diffusion, computes the mean squared error between the predicted and sampled noise, and updates the network. Validation uses the same objective without parameter updates.

In [ ]:
seed_everything(42, workers=True)

network = CompactUNet(
    in_channels=1,
    channels=(64, 128, 256),
    time_embedding_dim=64,
)
training_module = DDPMModule(
    model=network,
    diffusion=diffusion,
    learning_rate=1e-3,
)

trainer = Trainer(
    max_epochs=100,
    accelerator="auto",
    devices=4,
    default_root_dir=Path("../outputs/mojave"),
    log_every_n_steps=50,
)

In [ ]:
trainer.fit(training_module, datamodule=data)

## Load Checkpoint and generate samples

Sampling starts from Gaussian noise and applies the learned reverse process from the final diffusion timestep to timestep zero. With the configured schedule, this requires 1,000 U-Net evaluations for the generated batch. The generated values are clipped only for display.

In [ ]:
log_dir = Path("../outputs/mojave/lightning_logs/version_3")
ckpt_dir = Path("../outputs/mojave/lightning_logs/version_3/checkpoints/epoch=20-step=1407.ckpt")

In [ ]:
network = CompactUNet(
    in_channels=1,
    channels=(64, 128, 256),
    time_embedding_dim=64,
)

inference_module = DDPMModule.load_from_checkpoint(
    ckpt_dir,
    model=network,
    diffusion=diffusion,
    learning_rate=1e-3,
)
inference_module.eval()
inference_module.freeze()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# CSVLogger
df = pd.read_csv(log_dir / "metrics.csv")

print("Available metrics:", df.columns.tolist())

ax.set_facecolor("whitesmoke")

for metric, color, label in [
    ("train_loss_step", "steelblue", "Training loss (step)"),
    ("train_loss_epoch", "royalblue", "Training loss (epoch)"),
    ("val_loss", "coral", "Validation loss"),
]:
    values = df[["step", metric]].dropna()
    ax.plot(values["step"], values[metric], label=label, color=color, linewidth=1.8)

ax.set(
    title="DDPM Training Loss",
    xlabel="Training step",
    ylabel="Loss",
)

ax.grid(color="white", linewidth=1.2)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
generated_images = inference_module.diffusion.sample(
    model=inference_module.model,
    shape=(batch_size, 1, img_size, img_size),
)

In [ ]:
grid = make_grid(generated_images[:16], nrow=4)
preview = grid[0].cpu().numpy()  # Grayscale channel

# Power-law normalization
norm = PowerNorm(gamma=0.5, vmin=preview.min(), vmax=preview.max())

fig, ax = plt.subplots(figsize=(12, 8))

im = ax.imshow(preview, cmap="inferno", norm=norm)
fig.colorbar(im, ax=ax, label="Pixel intensity")

ax.axis("off")
None
preview.min(),preview.max(),preview.mean(),preview.std()